# Preference tuning with DPO, from scratch
A Bradley-Terry reward model, then DPO against a frozen reference, then a beta sweep.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
from prefs import FEATURES, THETA_REF, make_prompts, sample_pairs, true_reward
from dpo import train_reward_model, train_dpo, win_rate, kl_to_ref, expected_reward

## 1. Toy world + synthetic preferences

In [ ]:
rng = np.random.default_rng(42)
tr, te = make_prompts(400, 6, rng), make_prompts(300, 6, rng)
P, W, L, agree = sample_pairs(tr, THETA_REF, 3000, rng)
Pt, Wt, Lt, agree_t = sample_pairs(te, THETA_REF, 1500, rng)
dphi, dphi_t = tr[P, W] - tr[P, L], te[Pt, Wt] - te[Pt, Lt]
print('labels agree with true order:', agree_t.mean())

## 2. Bradley-Terry reward model

In [ ]:
w_rm, _ = train_reward_model(dphi, 500, 0.05)
print({f: round(float(v), 2) for f, v in zip(FEATURES, w_rm)})
print('test pairwise acc', ((dphi_t @ w_rm) > 0).mean())

## 3. DPO with a beta sweep
The margin is `beta * (theta - theta_ref) . (phi_w - phi_l)`: logZ cancels for a softmax policy.

In [ ]:
r_te = true_reward(te)
for beta in (0.1, 1.0, 10.0):
    th, hist = train_dpo(THETA_REF, dphi, beta, 3000, 0.05)
    print(f'beta={beta:5}: win={win_rate(th, THETA_REF, te, r_te):.4f}  KL={kl_to_ref(th, THETA_REF, te):.4f}  '
          f'E[r]={expected_reward(th, te, r_te):.3f}  loss={hist[-1]:.4f}')

## 4. DPO matches the closed-form RLHF optimum
`pi* = pi_ref * exp(r_rm / beta) / Z`, i.e. `theta* = theta_ref + w_rm / beta`

In [ ]:
th, _ = train_dpo(THETA_REF, dphi, 1.0, 3000, 0.05)
print('DPO theta     ', th.round(3))
print('RLHF optimum  ', (THETA_REF + w_rm / 1.0).round(3))

## 5. Full sweep
`python run_smoke.py` writes `results/`.

In [ ]:
import json
print(json.dumps(json.load(open('../results/metrics.json'))['headline'], indent=1))